# FedTalk Quick Start

This notebook demonstrates the core building blocks of the FedTalk package:
computing market price/volatility around an FOMC announcement, and getting an
LLM-based market reaction prediction from statement/news snippets.

Before running this, make sure you have:
- Installed dependencies: `pip install -r requirements.txt`
- Copied `src/fedtalk/api_keys.py.example` to `src/fedtalk/api_keys.py` and filled in your OpenAI/Pinecone keys
- Set `ALPACA_API_KEY` / `ALPACA_SECRET_KEY` environment variables

In [ ]:
import sys
sys.path.insert(0, "../src")

import datetime
from fedtalk.utils import finance_util

## Price change / volatility around an FOMC statement

`get_price_change` looks up cached daily price bars under `data/raw/data_1Min/price/`
(or fetches them from Alpaca if not cached) and returns the percentage change in SPY
between two timestamps.

In [ ]:
start = datetime.datetime(2024, 7, 31, 18, 30)
end = datetime.datetime(2024, 7, 31, 18, 35)

price_change = finance_util.get_price_change(start, end)
volatility = finance_util.get_price_volatility(start, end)

print(f"Price change: {price_change}")
print(f"Volatility: {volatility}")

## LLM market reaction prediction

`analysis_util.get_market_reaction_predictions` takes batches of statement/news
snippets (each a dict with `Id`, `Average Similarity Score`, `Price Movement`,
`Percent Change`) and returns the LLM's Positive/Negative prediction with reasoning.
Requires a valid `openai_api_key` in `src/fedtalk/api_keys.py`.

In [ ]:
from fedtalk.analysis import analysis_util

train_batch = [
    {"Id": 1, "Average Similarity Score": 0.82, "Price Movement": "Positive", "Percent Change": 0.0012},
]
test_batch = [
    {"Id": 2, "Average Similarity Score": 0.77, "Price Movement": None, "Percent Change": None},
]

predictions, metrics = analysis_util.get_market_reaction_predictions(train_batch, test_batch)
print(predictions)
print(metrics)